In [1]:
%config InlineBackend.figure_format = 'retina'

In [2]:
from common import *

default figsize: (3.3339558599695587, 2.0604980384530722)


In [3]:

df = read_trace("1.csv")
df.head(n=20)

,type,timestamp_us,count,layer_idx,seq_len,token_idx
0,StartTracing,0,<NA>,<NA>,<NA>,<NA>
1,CheckpointBegin,978199,<NA>,<NA>,<NA>,<NA>
2,CheckpointMetadata,978372,<NA>,<NA>,<NA>,<NA>
3,CheckpointFilesOpen,978406,<NA>,<NA>,<NA>,<NA>
4,CheckpointHCacheWrite,978584,<NA>,<NA>,<NA>,<NA>
5,CheckpointKVCacheWrite,979045,<NA>,<NA>,<NA>,<NA>
6,CheckpointFlush,979049,<NA>,<NA>,<NA>,<NA>
7,CheckpointFsyncData,982098,<NA>,<NA>,<NA>,<NA>
8,CheckpointWatermark,983734,<NA>,<NA>,<NA>,<NA>
9,CheckpointComplete,983735,0,<NA>,<NA>,<NA>


In [4]:
# Match checkpoint end events with their corresponding begin events
ends = df[df['type'] == 'CheckpointComplete'][['timestamp_us', 'count']].copy()
begins = df[df['type'] == 'CheckpointBegin'][['timestamp_us']].copy()
begins.columns = ['begin_timestamp']

# Merge to find most recent begin before each end
checkpoint_df = pd.merge_asof(ends, begins, left_on='timestamp_us', right_on='begin_timestamp', direction='backward')
checkpoint_df = checkpoint_df.rename(columns={'timestamp_us': 'end_timestamp'})
checkpoint_df['duration_us'] = checkpoint_df['end_timestamp'] - checkpoint_df['begin_timestamp']
checkpoint_df['event_type'] = 'checkpoint'
checkpoint_df = checkpoint_df[['begin_timestamp', 'end_timestamp', 'duration_us', 'count', 'event_type']]
checkpoint_df

,begin_timestamp,end_timestamp,duration_us,count,event_type
0,978199,983735,5536,0,checkpoint
1,24017458,24028084,10626,64,checkpoint
2,24065735,24075612,9877,32,checkpoint
3,24113196,24122524,9328,32,checkpoint
4,24159711,24170167,10456,32,checkpoint
...,...,...,...,...,...
72,27322884,27334358,11474,32,checkpoint
73,27373855,27383164,9309,32,checkpoint
74,27422786,27432208,9422,32,checkpoint
75,27471899,27481241,9342,32,checkpoint


In [5]:
# Match layer complete events with their corresponding begin events
layer_ends = df[df['type'] == 'LayerComplete'][['timestamp_us', 'layer_idx', 'seq_len', 'token_idx']].copy()
layer_begins = df[df['type'] == 'LayerBegin'][['timestamp_us', 'layer_idx', 'seq_len', 'token_idx']].copy()
layer_begins.columns = ['begin_timestamp', 'layer_idx', 'seq_len', 'token_idx']

# Merge to find corresponding begin for each end (matching on layer_idx, seq_len, token_idx)
layer_df = pd.merge_asof(layer_ends, layer_begins, 
                         left_on='timestamp_us', right_on='begin_timestamp', 
                         by=['layer_idx', 'seq_len', 'token_idx'], direction='backward')
layer_df = layer_df.rename(columns={'timestamp_us': 'end_timestamp'})
layer_df['duration_us'] = layer_df['end_timestamp'] - layer_df['begin_timestamp']
layer_df['event_type'] = 'layer'
layer_df = layer_df[['begin_timestamp', 'end_timestamp', 'duration_us', 'layer_idx', 'seq_len', 'token_idx', 'event_type']]
layer_df

,begin_timestamp,end_timestamp,duration_us,layer_idx,seq_len,token_idx,event_type
0,23965219,24017355,52136,0,32,0,layer
1,24028109,24065662,37553,1,32,0,layer
2,24075631,24113078,37447,2,32,0,layer
3,24122541,24159617,37076,3,32,0,layer
4,24170188,24208306,38118,4,32,0,layer
...,...,...,...,...,...,...,...
69,27283344,27322779,39435,13,32,64,layer
70,27334382,27373771,39389,14,32,64,layer
71,27383187,27422672,39485,15,32,64,layer
72,27432225,27471786,39561,16,32,64,layer


In [6]:
# Merge checkpoint and layer dataframes, sort by end_timestamp
merged_df = pd.concat([checkpoint_df, layer_df], ignore_index=True).sort_values('end_timestamp').reset_index(drop=True)
merged_df['duration_ms'] = merged_df['duration_us'] / 1000
merged_df.head(n=30)

,begin_timestamp,end_timestamp,duration_us,count,event_type,layer_idx,seq_len,token_idx,duration_ms
0,978199,983735,5536,0,checkpoint,<NA>,<NA>,<NA>,5.536
1,23965219,24017355,52136,<NA>,layer,0,32,0,52.136
2,24017458,24028084,10626,64,checkpoint,<NA>,<NA>,<NA>,10.626
3,24028109,24065662,37553,<NA>,layer,1,32,0,37.553
4,24065735,24075612,9877,32,checkpoint,<NA>,<NA>,<NA>,9.877
5,24075631,24113078,37447,<NA>,layer,2,32,0,37.447
6,24113196,24122524,9328,32,checkpoint,<NA>,<NA>,<NA>,9.328
7,24122541,24159617,37076,<NA>,layer,3,32,0,37.076
8,24159711,24170167,10456,32,checkpoint,<NA>,<NA>,<NA>,10.456
9,24170188,24208306,38118,<NA>,layer,4,32,0,38.118
